# Классификация сообщений мессенджера Авито
## Этап 6 — обучение модели

Обучение классификатора на 5 классов: `normal`, `external`, `spam`, `harassment`, `threat`.

### Требования из ТЗ, влияющие на решение

| Требование | Значение | Следствие для модели |
|---|---|---|
| Латентность инференса | ≤ 80 мс (средн. 50, p98 ≤ 100) | Лёгкая модель, CPU-инференс |
| Язык / стиль | русский, короткие сообщения, опечатки, сленг, завуалированные намерения | Нужны subword / символьные признаки |
| Precision класса `normal` | ≥ 0.95 | Не пропускать спам/угрозы как «обычное» |
| Recall по токсичным классам | максимально высокий | Ловить как можно больше нарушений |
| Дисбаланс | `normal` 92% → `threat` 0.07% | Взвешивание классов, аккуратные метрики |

### Критичность ошибок (из ТЗ)
- `normal` → токсичный (FN для normal): **критично** — валидное сообщение не доставлено, срыв сделки.
- токсичный → `normal`: пропуск нарушения, бьёт по precision normal.
- ошибка между токсичными классами: **менее критична**.


## 1. Обоснование выбора модели

Главное ограничение — **латентность ≤ 80 мс на CPU**. Оно отсекает тяжёлые
трансформеры и определяет двухуровневую стратегию.

| Модель | Параметры | CPU-латентность (короткий текст) | Русский | Вердикт |
|---|---|---|---|---|
| TF-IDF (char+word) + LogReg | — | < 2 мс | через символьные n-граммы | **Бейзлайн** |
| `cointegrated/rubert-tiny2` | 29M | ~10–30 мс | да (родной) | **Основная модель** |
| DeepPavlov/rubert-base | 180M | ~80–200 мс | да | не проходит p98 |
| mDeBERTa-v3-base | 280M | ~200–500 мс | да | только офлайн (этап 5) |

**Почему TF-IDF как бейзлайн.** Это «нетривиальный baseline» из ТЗ. Символьные
n-граммы (3–5) устойчивы к опечаткам и обфускации (`т*г`, `телега`, `т е л е г а`),
линейная модель даёт инференс < 2 мс и высокий precision. Это нижняя планка качества.

**Почему `rubert-tiny2` как основная.** Единственный трансформер, реально влезающий
в ≤ 80 мс на CPU. Обучен на русском, subword-токенизация устойчива к опечаткам,
контекстные эмбеддинги ловят завуалированную токсичность лучше мешка n-грамм.
ruBERT-base / mDeBERTa дают +1–2% качества, но не проходят по латентности
(mDeBERTa использовалась только офлайн для разметки на этапе 5 — там латентность не важна).

**Дисбаланс.** Взвешенный loss (обратная частота классов). Альтернативы —
oversampling редких классов, focal loss — вынесены в «следующие шаги».


## 2. Окружение и зависимости

Ноутбук рассчитан на Kaggle (GPU T4). Локально GPU не обязателен для бейзлайна.

In [ ]:
# Kaggle: раскомментировать при необходимости
# !pip install -q transformers datasets scikit-learn

import os, time, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (
    classification_report, confusion_matrix,
    precision_recall_fscore_support, ConfusionMatrixDisplay,
)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

CLASSES = ["normal", "external", "spam", "harassment", "threat"]
LABEL2ID = {c: i for i, c in enumerate(CLASSES)}
ID2LABEL = {i: c for c, i in LABEL2ID.items()}


In [ ]:
# Путь к данным: Kaggle-датасет либо локальная папка
KAGGLE_DIR = "/kaggle/input/avito-augmented"   # сюда загрузить train/val/test.csv как Kaggle Dataset
LOCAL_DIR  = "data/augmented"
DATA_DIR = KAGGLE_DIR if os.path.exists(KAGGLE_DIR) else LOCAL_DIR
print("DATA_DIR =", DATA_DIR)

train = pd.read_csv(f"{DATA_DIR}/train.csv")
val   = pd.read_csv(f"{DATA_DIR}/val.csv")
test  = pd.read_csv(f"{DATA_DIR}/test.csv")

for name, df in [("train", train), ("val", val), ("test", test)]:
    df["text"] = df["text"].fillna("").astype(str)
    df["y"] = df["label"].map(LABEL2ID)
    print(f"{name}: {len(df):,} строк | классы: {df['label'].value_counts().to_dict()}")


## 3. Данные

- **train** — аугментированный (реальные Авито + AlexSham + ru_paradetox + синтетика).
- **val / test** — **только реальные сообщения Авито** → честная оценка в проде.

> Важно: в val/test редкие классы представлены единичными примерами
> (`threat`≈1, `harassment`≈8 на 36k). Их recall будет статистически шумным —
> ориентируемся на `normal`-precision/recall (оцениваются надёжно) и на токсичные
> классы в совокупности. Для надёжной оценки редких классов в «следующих шагах»
> предлагается отдельный сбалансированный held-out.


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
train["label"].value_counts().reindex(CLASSES).plot.bar(ax=ax[0], color="#4C72B0")
ax[0].set_title("train (log)"); ax[0].set_yscale("log"); ax[0].set_ylabel("кол-во")
test["label"].value_counts().reindex(CLASSES).plot.bar(ax=ax[1], color="#55A868")
ax[1].set_title("test (log)"); ax[1].set_yscale("log")
for a in ax: a.tick_params(axis="x", rotation=30)
plt.tight_layout(); plt.show()


## 4. Бейзлайн: TF-IDF (char + word) + Logistic Regression

Символьные n-граммы (3–5) ловят опечатки и обфускацию, словные (1–2) — обычную
лексику. `class_weight="balanced"` компенсирует дисбаланс. Это нижняя планка
качества и почти нулевая латентность.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline, FeatureUnion

baseline = Pipeline([
    ("features", FeatureUnion([
        ("word", TfidfVectorizer(analyzer="word", ngram_range=(1, 2),
                                 min_df=3, max_features=100_000, sublinear_tf=True)),
        ("char", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5),
                                 min_df=3, max_features=200_000, sublinear_tf=True)),
    ])),
    ("clf", LogisticRegression(max_iter=1000, C=4.0,
                               class_weight="balanced", n_jobs=-1)),
])

t0 = time.time()
baseline.fit(train["text"], train["y"])
print(f"Обучение бейзлайна: {time.time()-t0:.1f} c")


In [ ]:
def report(y_true, y_pred, title):
    print(f"=== {title} ===")
    print(classification_report(y_true, y_pred, labels=range(len(CLASSES)),
                                target_names=CLASSES, digits=3, zero_division=0))
    p, r, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=range(len(CLASSES)), zero_division=0)
    ni = LABEL2ID["normal"]
    print(f"normal precision = {p[ni]:.3f}  (цель ≥ 0.95)")
    print(f"normal recall    = {r[ni]:.3f}  (FN normal = критичны)")
    return p, r, f1

val_pred = baseline.predict(val["text"])
_ = report(val["y"], val_pred, "Бейзлайн / val")


In [ ]:
# Латентность бейзлайна (одно сообщение, CPU)
sample = "привет, скинь пожалуйста номер телефона в тг"
for _ in range(20): baseline.predict([sample])
ts = []
for _ in range(200):
    t = time.perf_counter(); baseline.predict([sample]); ts.append((time.perf_counter()-t)*1000)
ts = np.array(ts)
print(f"Бейзлайн латентность (CPU): p50={np.percentile(ts,50):.2f} мс | "
      f"p98={np.percentile(ts,98):.2f} мс | mean={ts.mean():.2f} мс")


## 5. Основная модель: fine-tune `cointegrated/rubert-tiny2`

Дообучаем компактный русский BERT. Дисбаланс компенсируем взвешенным
кросс-энтропийным loss (веса обратно пропорциональны частоте класса).


In [ ]:
import torch
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer)
from datasets import Dataset

MODEL_NAME = "cointegrated/rubert-tiny2"
MAX_LEN = 128            # p95 длины ~160 символов → 128 токенов с запасом
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device =", device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def to_ds(df):
    ds = Dataset.from_pandas(df[["text", "y"]].rename(columns={"y": "labels"}),
                             preserve_index=False)
    return ds.map(lambda b: tokenizer(b["text"], truncation=True, max_length=MAX_LEN),
                  batched=True, remove_columns=["text"])

ds_train, ds_val, ds_test = to_ds(train), to_ds(val), to_ds(test)


In [ ]:
# Веса классов (обратная частота, нормированы)
counts = train["y"].value_counts().sort_index().values.astype(float)
class_weights = torch.tensor((counts.sum() / (len(counts) * counts)), dtype=torch.float)
print("Веса классов:", {CLASSES[i]: round(float(w), 2) for i, w in enumerate(class_weights)})

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(CLASSES), id2label=ID2LABEL, label2id=LABEL2ID)


In [ ]:
class WeightedTrainer(Trainer):
    def __init__(self, class_weights=None, **kw):
        super().__init__(**kw)
        self.class_weights = class_weights
    def compute_loss(self, model, inputs, return_outputs=False, **kw):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss_fct = torch.nn.CrossEntropyLoss(
            weight=self.class_weights.to(outputs.logits.device))
        loss = loss_fct(outputs.logits.view(-1, len(CLASSES)), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    p, r, f1, _ = precision_recall_fscore_support(
        labels, preds, labels=range(len(CLASSES)), average=None, zero_division=0)
    ni = LABEL2ID["normal"]
    return {"macro_f1": float(f1.mean()),
            "normal_precision": float(p[ni]),
            "normal_recall": float(r[ni])}


In [ ]:
from transformers import DataCollatorWithPadding
collator = DataCollatorWithPadding(tokenizer)

args = TrainingArguments(
    output_dir="rubert_tiny2_avito",
    num_train_epochs=3,
    per_device_train_batch_size=64,
    per_device_eval_batch_size=128,
    learning_rate=3e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy="epoch",          # старые версии transformers: evaluation_strategy
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    logging_steps=200,
    report_to="none",
)

import inspect
trainer_kwargs = dict(
    class_weights=class_weights,
    model=model, args=args,
    train_dataset=ds_train, eval_dataset=ds_val,
    data_collator=collator, compute_metrics=compute_metrics,
)
# Совместимость: transformers>=5 — processing_class, <5 — tokenizer
if "processing_class" in inspect.signature(Trainer.__init__).parameters:
    trainer_kwargs["processing_class"] = tokenizer
else:
    trainer_kwargs["tokenizer"] = tokenizer

trainer = WeightedTrainer(**trainer_kwargs)
trainer.train()


## 6. Оценка на test (реальные сообщения Авито)

In [ ]:
pred = trainer.predict(ds_test)
y_pred = np.argmax(pred.predictions, axis=-1)
p, r, f1 = report(test["y"].values, y_pred, "rubert-tiny2 / test")


In [ ]:
cm = confusion_matrix(test["y"], y_pred, labels=range(len(CLASSES)))
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(cm, display_labels=CLASSES).plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title("Confusion matrix — test"); plt.xticks(rotation=30); plt.tight_layout(); plt.show()

# Критичная ошибка: normal предсказан как токсичный (FN normal)
ni = LABEL2ID["normal"]
normal_as_toxic = cm[ni].sum() - cm[ni, ni]
print(f"normal ошибочно помечен как НЕ-normal: {normal_as_toxic} из {cm[ni].sum()} "
      f"({normal_as_toxic/cm[ni].sum()*100:.2f}%) — критичные ошибки (срыв сделки)")


In [ ]:
# Латентность основной модели на CPU (прод-условие — одно сообщение)
cpu_model = model.to("cpu").eval()
enc = tokenizer(sample, return_tensors="pt", truncation=True, max_length=MAX_LEN)
with torch.no_grad():
    for _ in range(10): cpu_model(**enc)
    ts = []
    for _ in range(200):
        t = time.perf_counter(); cpu_model(**enc); ts.append((time.perf_counter()-t)*1000)
ts = np.array(ts)
print(f"rubert-tiny2 латентность (CPU, 1 поток): "
      f"p50={np.percentile(ts,50):.1f} мс | p98={np.percentile(ts,98):.1f} мс | mean={ts.mean():.1f} мс")
print("Цель ТЗ: средн. ≤ 50 мс, p98 ≤ 100 мс")


In [ ]:
# Сохранение модели и токенизатора (для микросервиса / экспорта с Kaggle)
SAVE_DIR = "rubert_tiny2_avito_final"
cpu_model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print("Сохранено в", SAVE_DIR)
# На Kaggle: добавить папку в Output, затем скачать или подключить как Dataset

## 7. Сравнение и выводы

| Модель | macro-F1 | normal precision | латентность p98 (CPU) |
|---|---|---|---|
| TF-IDF + LogReg | _заполнить_ | _заполнить_ | < 2 мс |
| rubert-tiny2 | _заполнить_ | _заполнить_ | _заполнить_ |

(значения подставить после запуска на Kaggle)

**Ожидаемый итог:** rubert-tiny2 выигрывает по recall на скрытой токсичности и
по macro-F1, оставаясь в рамках латентности. TF-IDF — быстрый запасной вариант
и компонент возможного ансамбля.


## 8. Следующие шаги

1. **Латентность под прод.** Квантизация (`torch.quantization` / ONNX Runtime int8)
   для гарантии p98 ≤ 100 мс на проде, а не только на Kaggle-CPU.
2. **Политика решений.** Подбор порогов вероятности под бизнес-стоимость ошибок:
   доставить / доставить с предупреждением / заблокировать. Калибровка так, чтобы
   `normal`-precision ≥ 0.95 при максимальном recall токсичных.
3. **Редкие классы.** Отдельный сбалансированный held-out для надёжной оценки
   `threat`/`spam`; досбор данных угроз (этап 5 показал их дефицит).
4. **Дисбаланс.** Сравнить взвешенный loss с oversampling и focal loss.
5. **Сохранение модели** (`trainer.save_model`) и экспорт для микросервиса.
